In [1]:
import sys
from pathlib import Path
 
print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)
 
for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

Python: 3.12.9
Folder: mlops
numpy - ok
pandas - ok
sklearn - ok


# Build the dataset

In [2]:
import csv
from pathlib import Path
import numpy as np
 
SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)
 
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path
 
 
if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)


dataset ready: data/delivery_times.csv


# Loads the data, separates features (X) from the target (y), and does an 80/20 train/test split.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
 
orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X = orders[FEATURES]
y = orders["delivery_min"]
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), len(X_test))

480 120


# Define a function that trains a model, measures its error on both the training data and the test data, and computes the gap between them.

In [4]:
from sklearn.metrics import mean_absolute_error
 
def score_both_ways(model, name):
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    gap = test_mae - train_mae
    print(name, "train", round(train_mae, 2), "test", round(test_mae, 2), "gap", round(gap, 2))
    return {"name": name, "train": train_mae, "test": test_mae, "gap": gap}

# Model 1: LinearRegression

In [5]:
from sklearn.linear_model import LinearRegression
 
linear = score_both_ways(LinearRegression(), "LinearRegression")

LinearRegression train 2.04 test 1.92 gap -0.11


# Model 2: decision tree, no limit

In [6]:
from sklearn.tree import DecisionTreeRegressor
 
wild_tree = score_both_ways(DecisionTreeRegressor(random_state=42), "DecisionTree (no limit)")

DecisionTree (no limit) train 0.0 test 3.43 gap 3.43


# Model 3: shallow tree (depth 4)

In [7]:
small_tree = score_both_ways(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")

DecisionTree (depth 4) train 3.66 test 4.23 gap 0.57


# Model 4: RandomForest

In [8]:
from sklearn.ensemble import RandomForestRegressor
 
forest = score_both_ways(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

RandomForest (50 trees) train 1.0 test 2.39 gap 1.4


In [9]:
results = pd.DataFrame([linear, wild_tree, small_tree, forest]).round(2).sort_values("test")
print(results.to_string(index=False))


                   name  train  test   gap
       LinearRegression   2.04  1.92 -0.11
RandomForest (50 trees)   1.00  2.39  1.40
DecisionTree (no limit)   0.00  3.43  3.43
 DecisionTree (depth 4)   3.66  4.23  0.57


# Cross-validation - Trains and tests each model 5 times on 5 different data splits and averages the error

In [10]:
from sklearn.model_selection import cross_val_score
 
def cross_validate(model, name):
    scores = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(name, "MAE", round(scores.mean(), 2))
    return scores.mean()
 
cv_linear = cross_validate(LinearRegression(), "LinearRegression")
cv_tree = cross_validate(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")
cv_forest = cross_validate(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

LinearRegression MAE 2.03
DecisionTree (depth 4) MAE 4.57
RandomForest (50 trees) MAE 2.69


# Tries several tree depths (2, 3, 4, 6, 8, None), scores each with cross-validation, and finds the best one.

In [11]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}
 
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    scores = cross_val_score(tree, X, y, cv=5,
                             scoring="neg_mean_absolute_error")
    mae = -scores.mean()
    scores_by_depth[depth] = round(float(mae), 3)
 
best_depth = min(scores_by_depth, key=scores_by_depth.get)
print(scores_by_depth)
print("best depth:", best_depth)

{2: 5.858, 3: 4.85, 4: 4.573, 6: 3.642, 8: 3.395, None: 3.478}
best depth: 8


# Ranks the three model

In [12]:
ranking = sorted(
    {"LinearRegression": cv_linear, "DecisionTree(4)": cv_tree, "RandomForest(50)": cv_forest}.items(),
    key=lambda kv: kv[1],
)
for name, mae in ranking:
    print(name, round(mae, 2))

LinearRegression 2.03
RandomForest(50) 2.69
DecisionTree(4) 4.57


# To Do: Train Logistic Regression, Decision Tree, and Random Forest classifiers on the Breast Cancer dataset, compare their train/test accuracy, use cross-validation to find the best tree depth, and report the confusion matrix and classification metrics to identify the best-performing model. https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset

In [17]:
# ============================================================
# CLASSIFICATION ON DELIVERY TIME DATA
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

# Convert delivery time into two classes
# 0 = Fast delivery
# 1 = Slow delivery

median_time = y.median()
y_class = (y > median_time).astype(int)

print("Median delivery time:", median_time)
print("0 = Fast, 1 = Slow")


# Train/test split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)


# ============================================================
# TRAIN THREE CLASSIFIERS
# ============================================================

logistic = LogisticRegression(max_iter=1000)
tree = DecisionTreeClassifier(random_state=42)
forest = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)

models = {
    "Logistic Regression": logistic,
    "Decision Tree": tree,
    "Random Forest": forest
}

results = {}

for name, model in models.items():

    model.fit(X_train_c, y_train_c)

    train_pred = model.predict(X_train_c)
    test_pred = model.predict(X_test_c)

    results[name] = {
        "model": model,
        "train_accuracy": accuracy_score(y_train_c, train_pred),
        "test_accuracy": accuracy_score(y_test_c, test_pred),
        "predictions": test_pred
    }


# ============================================================
# TRAIN / TEST ACCURACY COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("TRAIN / TEST ACCURACY")
print("=" * 70)

for name, result in results.items():
    print(
        f"{name:20s} "
        f"Train: {result['train_accuracy']:.3f}  "
        f"Test: {result['test_accuracy']:.3f}"
    )


# ============================================================
# CONFUSION MATRIX + CLASSIFICATION METRICS
# ============================================================

print("\n" + "=" * 70)
print("CONFUSION MATRICES AND CLASSIFICATION METRICS")
print("=" * 70)

metric_results = []

for name, result in results.items():

    y_pred = result["predictions"]

    cm = confusion_matrix(y_test_c, y_pred)

    precision = precision_score(y_test_c, y_pred)
    recall = recall_score(y_test_c, y_pred)
    f1 = f1_score(y_test_c, y_pred)
    accuracy = accuracy_score(y_test_c, y_pred)

    print("\n" + "-" * 50)
    print(name)
    print("-" * 50)

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(
        y_test_c,
        y_pred,
        target_names=["Fast", "Slow"]
    ))

    metric_results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])


# ============================================================
# METRICS COMPARISON
# ============================================================

metrics_df = pd.DataFrame(
    metric_results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

print("\n" + "=" * 70)
print("CLASSIFICATION METRICS COMPARISON")
print("=" * 70)

print(metrics_df.round(3).to_string(index=False))


# ============================================================
# CROSS-VALIDATION FOR DECISION TREE DEPTH
# ============================================================

depths = [2, 3, 4, 5, 6, 8, 10, None]
depth_scores = {}

for depth in depths:

    dt = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    scores = cross_val_score(
        dt,
        X_train_c,
        y_train_c,
        cv=5,
        scoring="accuracy"
    )

    depth_scores[depth] = scores.mean()

best_depth = max(depth_scores, key=depth_scores.get)

print("\n" + "=" * 70)
print("DECISION TREE DEPTH CROSS-VALIDATION")
print("=" * 70)

for depth, score in depth_scores.items():
    print(f"Depth {str(depth):4s}: {score:.3f}")

print("\nBest Tree Depth:", best_depth)
print("Best CV Accuracy:", round(depth_scores[best_depth], 3))


# ============================================================
# OPTIMIZED DECISION TREE
# ============================================================

best_tree = DecisionTreeClassifier(
    max_depth=best_depth,
    random_state=42
)

best_tree.fit(X_train_c, y_train_c)

best_tree_pred = best_tree.predict(X_test_c)

print("\n" + "=" * 70)
print("OPTIMIZED DECISION TREE")
print("=" * 70)

print("\nAccuracy:",
      round(accuracy_score(y_test_c, best_tree_pred), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_c, best_tree_pred))

print("\nClassification Report:")
print(classification_report(
    y_test_c,
    best_tree_pred,
    target_names=["Fast", "Slow"]
))


# ============================================================
# FINAL BEST MODEL
# ============================================================

best_model = metrics_df.loc[
    metrics_df["F1 Score"].idxmax()
]

print("\n" + "=" * 70)
print("BEST PERFORMING MODEL")
print("=" * 70)

print("Model    :", best_model["Model"])
print("Accuracy :", round(best_model["Accuracy"], 3))
print("Precision:", round(best_model["Precision"], 3))
print("Recall   :", round(best_model["Recall"], 3))
print("F1 Score :", round(best_model["F1 Score"], 3))

Median delivery time: 46.5
0 = Fast, 1 = Slow

TRAIN / TEST ACCURACY
Logistic Regression  Train: 0.938  Test: 0.967
Decision Tree        Train: 1.000  Test: 0.925
Random Forest        Train: 1.000  Test: 0.925

CONFUSION MATRICES AND CLASSIFICATION METRICS

--------------------------------------------------
Logistic Regression
--------------------------------------------------

Confusion Matrix:
[[57  3]
 [ 1 59]]

Classification Report:
              precision    recall  f1-score   support

        Fast       0.98      0.95      0.97        60
        Slow       0.95      0.98      0.97        60

    accuracy                           0.97       120
   macro avg       0.97      0.97      0.97       120
weighted avg       0.97      0.97      0.97       120


--------------------------------------------------
Decision Tree
--------------------------------------------------

Confusion Matrix:
[[57  3]
 [ 6 54]]

Classification Report:
              precision    recall  f1-score   suppor